> **Sobre las cifras.** Lo que este notebook imprime al ejecutarse es ilustrativo. Las cifras citables del proyecto son las de `resultados/RESULTADOS.md` y `resultados/resultados_congelados.json` (las produce `06_congelar_resultados.ipynb`). La última celda de este notebook compara esta corrida con esos valores.

In [ ]:
# Preparación (no modificar): ubica la raíz del repo, la usa como directorio de trabajo
# y pone codigo/ en el path. Funciona igual abriendo el notebook desde notebooks/ o desde
# la raíz, y también ejecutándolo con codigo/run_nb.py.
import os, sys
_raiz = os.getcwd()
while not os.path.isdir(os.path.join(_raiz, "codigo")) and os.path.dirname(_raiz) != _raiz:
    _raiz = os.path.dirname(_raiz)
os.chdir(_raiz)
sys.path.insert(0, os.path.join(_raiz, "codigo"))
print("Raíz del repo:", _raiz)

## Configuración: SIN trim (decisión del 2026-09-17)

Este notebook corre **sin el recorte iterativo** del programa original. Es una desviación
deliberada: con 57,552 hogares el trim colapsa la varianza del regresor de utilidad y deja
el 65 % de los hogares con efectos ingreso planos, y el solver de Newton converge en el
61.9 % (rango de elasticidades [0.505, 1.248]; `trim_2022`).

Es también la configuración del análisis comparativo entre años. La réplica de fidelidad al
Gauss, con trim, está en `01_replica_2014.ipynb`.

# Actualización a ENIGH 2022 — Aradillas (2018)
## Poder de mercado y bienestar social en hogares mexicanos

Aplica a la ENIGH 2022 la metodología del estudio de COFECE replicada en
`01_replica_2014.ipynb`. El cálculo vive en `aradillas_core.py` y la carga de datos
en `datos_2022.py`; este notebook solo los conecta.

Se puede abrir desde `notebooks/` o desde la raíz del repo (la primera celda resuelve la ruta).

---

### Desviación deliberada: se corre SIN el trim iterado

El Gauss recorta el 1 % de cada cola de la utilidad **en cada una de las 16 iteraciones**
del bucle OLS, de forma acumulativa. En 2014 eso cuesta el 28 % de la muestra y no hace
daño. **En 2022 destruye la identificación**, y la razón es una interacción entre el trim y
el tamaño de muestra: con 8,940 hogares las colas de la utilidad se regeneran entre
iteraciones y su varianza sobrevive; con 57,552 los cuantiles son estables, el recorte
muerde siempre en el mismo lugar y la varianza colapsa.

| (`resultados/RESULTADOS.md`) | 2014 con trim (`replica_2014`) | 2022 con trim (`trim_2022`) | 2022 sin trim (`comparable_2022`) |
|---|---|---|---|
| desviación estándar de `util` en el bucle | 1.170 | **0.425** | 0.770 |
| hogares con `f'(u0)` plano (`\|f'\|<0.3`) | 7.0 % | **65.2 %** | 36.8 % |
| convergencia del solver de Newton | 97.1 % | **61.9 %** | 86.1 % |
| rango de elasticidades | [0.436, 1.737] | **[0.505, 1.248]** | [0.554, 1.657] |

La cadena: el trim aplana la varianza del regresor `util` → los coeficientes en `u`
(los efectos ingreso) quedan mal identificados → la función de costo es casi plana donde
arranca Newton → la raíz cae lejos del punto de arranque → el solver converge en menos hogares y el
sistema deja de producir una cifra de bienestar utilizable (ver `resultados/RESULTADOS.md`).

**Por qué desviarse es lo correcto aquí:** el criterio de fidelidad al Gauss aplica a la
*réplica* de 2014, donde hay un programa original contra el cual contrastar. La
actualización a 2022 es trabajo nuevo, y conservar un mecanismo que demostrablemente
destruye la identificación en esta muestra no sería fidelidad. **Queda documentado como
hallazgo: el algoritmo del estudio original no escala a muestras del tamaño de la ENIGH
moderna.**

### Diferencias de datos respecto de 2014

* Hogar = `folioviv` + `foliohog` (una vivienda puede alojar varios hogares).
* Claves de producto alfanuméricas (`A004`) en vez de numéricas (`1004`). La conversión es
  determinista y las categorías se derivan de la tabla verificada de `datos_2014.py`, lo
  que evita el error de `productos.py` que ponía el aguacate en Frutas y omitía la pera.
* Precios de referencia de **2018**, deflactados a la ventana ago–nov 2022 (factor mediano
  ≈ 1.37). Materiales va por INPP, que le da la variación entre ciudades.
* **`ing_mon` no existe**: la ENIGH 2022 "Nueva serie" dejó de publicar el ingreso no
  monetario por separado. Se reconstruye sumando `ingresos.csv`, que **ya contiene solo
  ingreso monetario** — el componente no monetario (`estim_alqu`) es imputado y no aparece
  ahí. Verificado: Σ claves = \$53,929 ≈ `ing_cor − estim_alqu − remu_espec` = \$53,694.

In [ ]:
import sys
sys.path.insert(0, "codigo")
import numpy as np

import datos_2022
from aradillas_core import (estimar_easi, reconstruir_matrices, ModeloEASI,
                            demandas_marshallianas, elasticidades,
                            estimar_markups, variacion_equivalente,
                            cuadro_10, gini, sectores_significativos)

DATA_DIR = "datos/Data_2022/"
print("Módulos cargados.")

## 1–2. Precios locales y microdatos ENIGH 2022

Cascada esperada: 90,102 → 57,989 (filtros del Gauss) → 57,989 (400 km) → 57,552
(al menos una categoría con gasto ≥ 10).

El filtro de 400 km no descarta a nadie, y **no es un error**: la ENIGH 2022 cubre 90 mil
hogares con mucha mayor dispersión geográfica y prácticamente todos caen dentro de ese
radio de alguna de las 46 ciudades. En 2014 ese filtro solo quitaba 131 de 12,592.

In [ ]:
datos = datos_2022.cargar(DATA_DIR)

## 4–5. Sistema EASI y utilidad indirecta

`aplicar_trim=False` — ver la justificación en la portada.

In [ ]:
res = estimar_easi(datos.precios_ln, datos.w, datos.gasto_total, datos.Z,
                   n_cat=datos.n_cat, aplicar_trim=False)
datos = datos.submuestra(res["mask"])

mats = reconstruir_matrices(res["beta"], n_cat=datos.n_cat)
modelo = ModeloEASI(**mats)
epsilon = res["epsilon"]

util = modelo.utilidad_indirecta(datos.precios_ln, datos.Z, epsilon,
                                 datos.w, datos.gasto_total)

## 6. Elasticidades

In [ ]:
demandas, _ = demandas_marshallianas(
    modelo, datos.precios_ln, datos.Z, util, epsilon,
    datos.gasto_total, datos.factor_expansion, datos.n_cat)

elastic_nac, elastic_cd = elasticidades(
    modelo, datos.precios_ln, datos.Z, epsilon, datos.w, datos.gasto_total,
    datos.factor_expansion, demandas, datos.ciudad, datos.n_ciudades,
    datos.n_cat, util, nombres=datos.nombres_cat)

In [ ]:
paper14 = {'Tortillas':1.054,'Pan':1.462,'Pollo+Huevo':1.261,'Carne res':0.735,
           'Carnes proc.':0.968,'Lácteos':1.289,'Frutas':1.415,'Verduras':1.389,
           'Bebidas':1.110,'Medicamentos':0.943,'Transporte foráneo':0.847,
           'Materiales':0.934}

print("=== ELASTICIDADES 2022 ===")
print(f"{'Categoría':<22} {'2022':>8} {'paper 2014':>11} {'dif':>8}")
print("-"*52)
for j, n in enumerate(datos.nombres_cat):
    e = abs(elastic_nac[j]); p = paper14.get(n, np.nan)   # Pan de caja: sin cifra en el paper
    print(f"  {n:<20} {e:>8.3f} {p:>11.3f} {e-p:>8.3f}")
print(f"\nrango: [{np.abs(elastic_nac).min():.3f}, {np.abs(elastic_nac).max():.3f}]"
      f"   (paper 2014: [0.735, 1.462])")

## 7. Markups y poder de mercado (NEIO)

**Cómo leer el cuadro.** Si las elasticidades estuvieran comprimidas hacia −1, entonces
η ≈ p y la regresión devolvería β ≈ 1 con estadísticos t enormes por colinealidad casi
perfecta — una identidad algebraica, no poder de mercado. Que los β caigan en rango
plausible, que los t sean moderados y que **algunos sectores salgan no significativos** es
la señal de que la estimación es sana.

In [ ]:
P_cat = datos.precios_por_ciudad()
mk = estimar_markups(P_cat, elastic_cd, datos.vars_costos)
beta_eta, t_eta = mk["beta_eta"], mk["t_eta"]

paper_b = {'Tortillas':0.183,'Pan':1.477,'Pollo+Huevo':0.139,'Carne res':0.047,
           'Carnes proc.':0.017,'Lácteos':0.626,'Frutas':1.120,'Verduras':0.328,
           'Bebidas':0.047,'Medicamentos':0.026,'Transporte foráneo':0.081,
           'Materiales':0.493}

print("=== CUADRO 8: PODER DE MERCADO β_η ===")
print(f"{'Categoría':<22} {'β 2022':>8} {'t':>8} {'β paper14':>10}")
print("-"*52)
for j, n in enumerate(datos.nombres_cat):
    sig = "***" if abs(t_eta[j]) >= 2.326 else ("**" if abs(t_eta[j]) >= 1.645 else "   ")
    print(f"  {n:<20} {beta_eta[j]:>8.3f} {t_eta[j]:>8.2f} {paper_b.get(n, np.nan):>10.3f} {sig}")

## 8. Variación equivalente y bienestar

El denominador de la VE es `ing_cor` (la ENIGH 2022 "Nueva serie" ya no publica `ing_total`). El Gini se
calcula como en el Gauss —sobre la muestra completa del concentrado y con ajuste
multiplicativo por decil—, aquí con el ingreso monetario reconstruido desde `ingresos.csv`.

**Advertencia sobre magnitudes en pesos.** Las magnitudes en pesos de 2022 no están validadas
(falta el ajuste a pesos constantes); usar porcentajes. Las cifras vigentes están en
`resultados/RESULTADOS.md`.

El Gini observado de 2022 **no es directamente comparable** con el de 2014: son bases de
ingreso distintas, no desigualdades distintas.

In [ ]:
sig_95 = sectores_significativos(t_eta, beta_eta)   # V1: t >= 2.326
print(f"Sectores significativos: {int(sig_95.sum())} de {datos.n_cat}")

VE = variacion_equivalente(modelo, datos.precios_ln, datos.Z, epsilon, datos.w,
                           datos.gasto_total,
                           mk["markup_lerner"][datos.ciudad],   # V2: Lerner
                           sig_95)
c10 = cuadro_10(VE, datos.ingreso_cor)  # V3: ing_cor — la ENIGH 2022 "Nueva serie" ya no publica ing_total

paper_p = [30.9,23.6,21.4,18.9,16.7,15.1,13.6,11.9,9.5,5.7,15.7]
print("\n=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===")
print(f"{'Decil':<6} {'VE($)':>9} {'VE/Ing%':>9} {'paper14%':>9}")
print("-"*38)
for f in c10["deciles"]:
    print(f"  {f['decil']:<4} {f['VE']:>9.0f} {f['pct']:>9.1f} {paper_p[f['decil']-1]:>9.1f}")
print(f"  {'Tot':<4} {c10['total']['VE']:>9.0f} {c10['total']['pct']:>9.1f} {paper_p[-1]:>9.1f}")
print(f"\nRegresividad D1/D10: {c10['regresividad']:.2f}  (paper 2014: 4.42)")

g = gini(datos.ingreso_mon_completo, c10["tasas"])
print(f"\nGini observado:     {g['observado']:.3f}   [N={g['n']}]")
print(f"Gini contrafactual: {g['contrafactual']:.3f}")
print(f"Reducción:          {g['reduccion_pct']:.1f}%  (paper 2014: 7.3%)")

In [ ]:
# Cotejo con el congelado: esta corrida contra resultados/resultados_congelados.json
import json
_ref = json.load(open("resultados/resultados_congelados.json"))
_b = _ref["comparable_2022"]["markups"]["ciudad"]["bienestar"]["por_denominador"]["ing_cor"]
def _cotejo(filas, tol):
    print(f"{'cifra':<28}{'esta corrida':>14}{'congelado':>12}")
    ok = True
    for nombre, mio, cong in filas:
        bien = abs(mio - cong) <= tol[nombre]
        ok &= bien
        print(f"{nombre:<28}{mio:>14.4f}{cong:>12.4f}  {'✓' if bien else '✗ DIFERENTE'}")
    print("\nTodo coincide con el congelado." if ok else
          "\nHay diferencias: revisar versiones (numpy 1.26.4, pandas 2.2.3, scipy 1.13.1) y los datos.")
_cotejo([("VE / ingreso corriente (%)", c10["total"]["pct"], _b["ve_pct_total"]),
         ("regresividad D1/D10", c10["regresividad"], _b["regresividad"]),
         ("Gini obs. (ing_mon reconstruido)", g["observado"], _b["gini"]["ing_mon"]["observado"]),
         ("Gini contrafactual", g["contrafactual"], _b["gini"]["ing_mon"]["contrafactual"]),
         ("reducción del Gini (%)", g["reduccion_pct"], _b["gini"]["ing_mon"]["reduccion_pct"])],
        {"VE / ingreso corriente (%)": 0.011, "regresividad D1/D10": 0.011, "Gini obs. (ing_mon reconstruido)": 0.0011,
         "Gini contrafactual": 0.0011, "reducción del Gini (%)": 0.011})
